# 实验 2　文本清洗与按回切分

## 任务要求

1. 去除与正文无关的信息，统一空白符和标点，保留清洗规则。
2. 用正则表达式识别“第一回、第二回……”等回目标记，把文本切成章节。
3. 每章至少保存 3 个字段：章号、回目标题、正文内容。
4. 随机抽查 2—3 章验证切分，并记录出现错误时如何修正。

## 清洗规则

- 去掉 UTF-8 BOM；
- 统一换行符为 `\n`，全角空格替换为半角空格；
- 删除第 40 回结尾的目录式残留 `#第四十一回…`；
- 合并连续空格与空行。

## 切分规则

回目标题正则 `^[ \t]*第[一二三四五六七八九十百千零0-9]+回` **只匹配行首**，避免正文中引用“第 X 回”造成误切分。

In [1]:
# 让 notebook 在 notebooks/ 或仓库根目录下都能找到 src/ 包
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "sanguo_nlp").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sanguo_nlp.paths import FIGURES_DIR, TABLES_DIR, ensure_output_dirs

ensure_output_dirs()
print("已定位仓库根目录:", ROOT.name)

已定位仓库根目录: three-kingdoms-nlp-lab


In [2]:
from sanguo_nlp.corpus import clean_text, read_raw, split_chapters

raw = read_raw()
cleaned = clean_text(raw)

print("清洗前字符数:", len(raw))
print("清洗后字符数:", len(cleaned))
print("删除的字符数:", len(raw) - len(cleaned))
print("\n清洗后前 200 字:")
print(cleaned[:200])

清洗前字符数: 602415
清洗后字符数: 602151
删除的字符数: 264

清洗后前 200 字:
三国演义 上卷
罗贯中
滚滚长江东逝水，浪花淘尽英雄。是非成败转头空。青山依旧在，几度夕阳红。
白发渔樵江渚上，惯看秋月春风。一壶浊酒喜相逢。古今多少事，都付笑谈中。
--调寄《临江仙》
第一回 宴桃园豪杰三结义 斩黄巾英雄首立功
话说天下大势，分久必合，合久必分。周末七国分争，并入于秦。及秦灭之后，楚、汉分争，又并入于汉。汉朝自高祖斩白蛇而起义，一统天下，后来光武中兴，传至献帝，遂分为三国。推其


In [3]:
chapters = split_chapters(raw)
print("切分得到的章节数:", len(chapters))
print("字段:", list(chapters.columns))
chapters.head(3)

切分得到的章节数: 120
字段: ['章号', '回目', '标题', '正文']


,章号,回目,标题,正文
0,1,第一回,宴桃园豪杰三结义 斩黄巾英雄首立功,话说天下大势，分久必合，合久必分。周末七国分争，并入于秦。及秦灭之后，楚、汉分争，又并入于汉...
1,2,第二回,张翼德怒鞭督邮 何国舅谋诛宦竖,且说董卓字仲颖，陇西临洮人也，官拜河东太守，自来骄傲。当日怠慢了玄德，张飞性发，便欲杀之。玄...
2,3,第三回,议温明董卓叱丁原 馈金珠李肃说吕布,"且说曹操当日对何进曰：""宦官之祸，古今皆有；但世主不当假之权宠，使至于此。若欲治罪，当除元恶..."


In [4]:
chapters.to_csv(TABLES_DIR / "chapters.csv", index=False, encoding="utf-8-sig")
print("已保存 outputs/tables/chapters.csv")

已保存 outputs/tables/chapters.csv


## 随机抽查

随机抽取 3 回，核对回目标题、正文开头与结尾是否衔接自然。

In [5]:
import random

random.seed(42)
for i in random.sample(range(len(chapters)), 3):
    row = chapters.iloc[i]
    print(f"第{row['章号']}回 {row['标题']}")
    print("前 80 字:", row["正文"][:80])
    print("后 40 字:", row["正文"][-40:])
    print("-" * 60)

第82回 孙权降魏受九锡 先主征吴赏六军
前 80 字: 却说章武元年秋八月，先主起大军至夔关，驾屯白帝城。前队军马已出川口。近臣奏曰："吴使诸葛瑾至。"先主传旨教休放入。黄权奏曰："瑾弟在蜀为相，必有事而来。陛下何故
后 40 字: 来助黄忠。正是：老臣素矢忠君志，年少能成报国功。未知黄忠此去如何，且看下文分解。
------------------------------------------------------------
第15回 太史慈酣斗小霸王 孙伯符大战严白虎
前 80 字: 却说张飞拔剑要自刎，玄德向前抱住，夺剑掷地曰："古人云：'兄弟如手足，妻子如衣服。衣服破，尚可缝；手足断，安可续？'吾三人桃园结义，不求同生，但愿同死。今虽失了
后 40 字: 备即日就擒。"正是：不去江东图虎豹，却来徐郡斗蛟龙。不知其计若何，且听下文分解。
------------------------------------------------------------
第4回 废汉帝陈留践位 谋董贼孟德献刀
前 80 字: 且说董卓欲杀袁绍，李儒止之曰："事未可定，不可妄杀。"袁绍手提宝剑，辞别百官而出，悬节东门，奔冀州去了。卓谓太傅袁隗曰："汝侄无礼，吾看汝面，姑恕之。废立之事若
后 40 字: 来杀曹操。正是：设心狠毒非良士，操卓原来一路人。毕竟曹操性命如何，且听下文分解。
------------------------------------------------------------


In [6]:
assert list(chapters["章号"]) == list(range(1, len(chapters) + 1)), "章号不连续"
assert (chapters["正文"].str.len() > 0).all(), "存在空正文章节"
assert chapters["回目"].str.match(r"^第[一二三四五六七八九十百千零0-9]+回$").all(), "回目标记异常"

shortest = chapters.loc[chapters["正文"].str.len().idxmin()]
longest = chapters.loc[chapters["正文"].str.len().idxmax()]
print("校验通过：章号连续、正文非空、回目标记规范")
print(f"最短章：第{shortest['章号']}回，{len(shortest['正文'])} 字")
print(f"最长章：第{longest['章号']}回，{len(longest['正文'])} 字")

校验通过：章号连续、正文非空、回目标记规范
最短章：第24回，2775 字
最长章：第40回，10495 字


## 实验记录与讨论

- “看起来能切开”不等于“切分正确”，因此增加了三项断言：章号必须连续、正文不能为空、回目必须形如“第X回”。
- 如果文本开头有序言、目录或版权信息，应先用专门的正则（例如 `目录|版权|序言`）识别并删除，再按回切分。
- 本实验未出现报错；AI 工具主要用于提供正则表达式写法，随后通过自行运行与抽样抽查完成验证。